# 0 · Data preparation

**HyperSpot tutorial: Human Kidney Cancer, tumour core (patient PD47171).**

This notebook prepares the three inputs Spotiphy needs:
1. a single-cell RNA-seq **reference** (one patient, Ensembl gene IDs),
2. the **Visium** spots of the same patient's tumour core,
3. the matching **H&E image**.

Kernel: `HyperSpot` environment. Next: `01 Spotiphy Deconvolution`.

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc

from hyperspot.preprocessing import common, image, kidneycancer

from pathlib import Path

# All tutorial outputs live next to these notebooks; run the notebooks from tutorial/.
DATA_DIR      = Path("../demo/data_repo/Human Kidney Cancer")     # raw inputs (see README)
PATIENT_ID    = "PD47171"
PROJECT       = "Human Kidney Cancer Core"
SPOTIPHY_DIR  = Path("spotiphy_project_files") / PROJECT          # Spotiphy inputs/outputs
HYPERSPOT_DIR = Path("hyperspot_project_files") / PROJECT         # HyperSpot project (the CLI resolves
                                                                  # --project relative to ./hyperspot_project_files)
SPOTIPHY_DIR.mkdir(parents=True, exist_ok=True)

## Load the scRNA-seq reference and the Visium data

In [ ]:
adata_sc = sc.read_h5ad(DATA_DIR / "RCC_upload_final_raw_counts.h5ad")
adata_st = sc.read_h5ad(DATA_DIR / "visium_merge_core_upload.h5ad")

Keep only patient PD47171 in both datasets.

In [ ]:
adata_sc = kidneycancer.extract_sc_patient_data(adata_sc, PATIENT_ID)
adata_st = kidneycancer.extract_st_patient_data(adata_st, PATIENT_ID)
adata_sc, adata_st

The reference uses `broad_type` as the cell-type annotation (12 types).

In [ ]:
adata_sc.obs.groupby(["broad_type", "annotation"], observed=True).size().rename("cells").reset_index()

## Use Ensembl gene IDs in both datasets

The scRNA-seq reference is indexed by gene symbol, the Visium data by Ensembl ID. We map symbols to
Ensembl IDs with a GRCh38 annotation, then resolve the few symbols that map ambiguously.

In [ ]:
geneset = common.get_annotation(host="dec2021.archive.ensembl.org",
                                cache_filename="GRCh38-2020-A_annotations.json")
kidneycancer.find_conflicting_genes(adata_sc, adata_st, geneset)

In [ ]:
# Resolutions for the conflicts listed above
drop_cols = ["ENSG00000286070",   # GGT1 duplicate
             "ENSG00000280987",   # MATR3 duplicate
             "ENSG00000285053"]   # TBCE duplicate
rename_cols = {"ENSG00000284024": "MSANTD7",
               "ENSG00000269226": "TMSB15C"}

adata_sc, adata_st = kidneycancer.use_ensembl_gene_ids(adata_sc, adata_st, geneset,
                                                       drop_cols=drop_cols, rename_cols=rename_cols)

## Extract the H&E image

In [ ]:
img = image.get_image(adata_st, scale=True)
plt.imshow(img); plt.axis("off"); plt.show()

## Save the Spotiphy inputs

In [ ]:
adata_sc.write(SPOTIPHY_DIR / "Human Kidney Cancer scRNA Ref.h5ad")
adata_st.write(SPOTIPHY_DIR / f"{PROJECT}.h5ad")
img.save(SPOTIPHY_DIR / f"{PROJECT}.tif")
sorted(p.name for p in SPOTIPHY_DIR.iterdir())